##Project Overview

###Objective
**Second Brain** is designed to help you consume the large documents (papers, financal reports, market reports, etc.) and increase your productticity. It leverage the LLM to consume the documents in Google Drives Folders and answer the inqueries.

Smart Assistant
is to create a second brain which can help us consume the large documents and increase the productivity.

###Challenges

* Many papers and reports are PDF format, it is not easy to transform them to a structure data and search the answers.
* The answser may not comes from a single file but the combination of the multiple files

###Recognition
This works is inspired by [How to build a ChatGPT + Google Drive app with LangChain and Python](https://www.haihai.ai/gpt-gdrive/?ref=emergentmind)

### Wroking Environment and Preparation

* Copy this notebook to your account
* Make sure the Vertex AI API are enabled and billable
* You can use the exsiting [Google Folder (with files downloaded)](https://drive.google.com/drive/folders/1PgSIiyarwZmE96dA-QVzqITv39YGvdKk), the ID already assgined in this sample code. Or create your own foler to store the documents. Following are PDF files need to use for this demo.
  * Alphabet Annual Report: [2021](https://www.abc.xyz/assets/9a/bd/838c917c4b4ab21f94e84c3c2c65/goog-10-k-q4-2022.pdf), [2022](https://www.abc.xyz/assets/d9/85/b7649a9f48c4960adbce5bd9fb54/20220202-alphabet-10k.pdf)
  * Amazon Annual Report: [2021](https://s2.q4cdn.com/299287126/files/doc_financials/2022/ar/Amazon-2021-Annual-Report.pdf), [2022](https://s2.q4cdn.com/299287126/files/doc_financials/2023/ar/Amazon-2022-Annual-Report.pdf)
* Assigned the variable of the demo environment in follow session.

Identify the Project ID for Vertex AI API, Location, and Google Drive Folder ID.

In [ ]:
# Replace these to your own

PROJECT_ID = "1PgSIiyarwZmE96dA-QVzqITv39YGvdKk"  # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"}
FOLDER_ID = "" # @param {type:"string"}

In [ ]:
from platform import python_version
print(python_version())

In [ ]:
# Install Vertex AI LLM SDK, langchain and dependencies
# Remember to click RESTART RUNTIME after this step.
! pip install config --upgrade --user
! pip install google-cloud-aiplatform google-api-python-client

In [ ]:
# Install langchain and dependencies
! pip install langchain chromadb pypdf2

### Authenticating your notebook environment

It will pop up the authorization window for the Google Drive access.

In [ ]:
from google.colab import auth as google_auth
google_auth.authenticate_user()

### Import libraries

In [ ]:
import langchain
import vertexai

# Vertex AI
from google.cloud import aiplatform
from langchain.llms import VertexAI

vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"LangChain version: {langchain.__version__}")
print(f"Vertex AI SDK version: {aiplatform.__version__}")

Load the documents and store into the Chroma Database

In [ ]:
from langchain.document_loaders import GoogleDriveLoader

loader = GoogleDriveLoader(
    folder_id=FOLDER_ID,
    recursive=False
)
docs = loader.load()

In [ ]:
# split the documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
docs = text_splitter.split_documents(docs)

print(f"# of chunks = {len(docs)}")

In [ ]:
from langchain.embeddings import VertexAIEmbeddings
from langchain.vectorstores import Chroma

# Embedding
embeddings = VertexAIEmbeddings()
db = Chroma.from_documents(docs, embeddings)

In [ ]:
# Create chain to answer questions
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM model
llm = VertexAI(
    model_name="text-bison@001",
    max_output_tokens=256,
    temperature=0,
    top_p=0.95,
    top_k=40,
    verbose=True,
)

prompt_template = """
You are a trustworthy consultant.
Answer the users question based on the context only.
Do not make up data. If you cannot find answer in the context, then say 'Sorry, I cannot find the answers in the documents.'

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template = prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

# Seting the approriate retriver
#retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever = db.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4, "k": 4})

qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs,
)

###Asking Questions###

#### Asking questions which can be found in the single file.
Alphabet Net Income
* 2022: \$59,972 millions (in 2022 report)
* 2021: \$76,033 millions (in 2022, 2021 report)
* 2020: \$40,269 millions (in 2022, 2021 report)
* 2019: \$34,343 millions (in 2021 report)

In [ ]:
query = "What is the net income of Alphabet in 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

#### Asking questions which can be found in the multiple files.

Alphabet Net Income
* 2022: \$59,972 millions (in 2022 report)
* 2021: \$76,033 millions (in 2022, 2021 report)
* 2020: \$40,269 millions (in 2022, 2021 report)
* 2019: \$34,343 millions (in 2021 report)

In [ ]:
query = "List the net income of Alphabet in 2019 to 2022 in millions"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Google Cloud Revenue:
* 2022: \$26,280 millions (in 2022 report)
* 2021: \$19,206 millions (in 2021 report)

In [ ]:
query = "What's the revenue of Google Cloud in 2021 and 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

Amazon Employee Number:
* Dec 31, 2022: 1,541,000 (in 2022 report)
* Dec 31, 2021: 1,608,000 (in 2021 report)

In [ ]:
query = "How many employee of Amazon in end of 2021 and 2022?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])

#### Asking somthing which cannot find in the folder.

In [ ]:
query = "The distance between Sun and Earth?"
result = qa({"query": query})

print(f"Query: ", result["query"])
print(f"Answer: ", result["result"])
print(f"Source: ", result["source_documents"])